In [1]:
from pyspark.sql import SparkSession
import json, time

spark = (SparkSession.builder
    .appName("UrgenceNet-Partie2-RDD")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print(sc.version, sc.master, sc.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 22:27:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.3 spark://spark-master:7077 2


In [2]:
PATH = "s3a://lake/bronze/history/urgences_history.jsonl"

lignes = sc.textFile(PATH)                       # transformation (lazy)
events = lignes.map(json.loads).cache()          # parsing json.loads
print("Événements :", events.count())            # action : déclenche la lecture

care = events.filter(lambda e: e["status"] == "CARE_STARTED")
print("Prises en charge :", care.count())

26/11/05 22:27:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Événements : 50000


[Stage 1:>                                                          (0 + 2) / 2]

Prises en charge : 12492


In [3]:
t0 = time.time()
par_ville = (care.map(lambda e: (e["city"], 1))
                 .reduceByKey(lambda a, b: a + b))
res_count = sorted(par_ville.collect())
print("Durée : %.2f s" % (time.time() - t0))
for v, n in res_count:
    print(f"{v:<12} {n}")

[Stage 3:>                                                          (0 + 2) / 2]

Durée : 2.79 s
Casablanca   2814
Marrakech    3331
Rabat        3191
Tanger       3156


In [4]:
t0 = time.time()
somme_nb = care.map(lambda e: (e["city"], (e["wait_minutes"], 1))) \
               .aggregateByKey((0, 0),
                               lambda acc, v: (acc[0] + v[0], acc[1] + v[1]),   # dans la partition
                               lambda a, b: (a[0] + b[0], a[1] + b[1]))         # entre partitions
moy = sorted(somme_nb.mapValues(lambda s: (s[1], round(s[0] / s[1], 2))).collect())
print("Durée : %.2f s" % (time.time() - t0))
print(f"{'Ville':<12} {'Prises en charge':>17} {'Attente moy. (min)':>19}")
for v, (n, m) in moy:
    print(f"{v:<12} {n:>17} {m:>19}")

[Stage 4:>                                                          (0 + 2) / 2]

Durée : 1.48 s
Ville         Prises en charge  Attente moy. (min)
Casablanca                2814               34.86
Marrakech                 3331               34.42
Rabat                     3191               34.11
Tanger                    3156               34.96


In [5]:
tri = (events.filter(lambda e: e["status"] == "TRIAGED")
             .map(lambda e: (e["priority"], 1))
             .reduceByKey(lambda a, b: a + b)
             .sortByKey())
for p, n in tri.collect():
    print("Priorité", p, ":", n)

[Stage 12:>                                                         (0 + 2) / 2]

Priorité 1 : 804
Priorité 2 : 1979
Priorité 3 : 4269
Priorité 4 : 3622
Priorité 5 : 1837


In [6]:
top5 = (care.map(lambda e: (e["facility_id"], 1))
            .reduceByKey(lambda a, b: a + b)
            .takeOrdered(5, key=lambda x: -x[1]))
for f, n in top5:
    print(f, n)

[Stage 13:>                                                         (0 + 2) / 2]

F-011 1126
F-021 1059
F-001 960
F-031 951
F-023 608


In [7]:
print(par_ville.toDebugString().decode("utf-8"))

(2) PythonRDD[9] at collect at /tmp/ipykernel_348/2822284258.py:4 []
 |  MapPartitionsRDD[8] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[7] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[6] at reduceByKey at /tmp/ipykernel_348/2822284258.py:3 []
    |  PythonRDD[5] at reduceByKey at /tmp/ipykernel_348/2822284258.py:3 []
    |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 1673.5 KiB; DiskSize: 0.0 B
    |  s3a://lake/bronze/history/urgences_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  s3a://lake/bronze/history/urgences_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


In [9]:
# Pipeline complet SANS cache, depuis le fichier (comparaison équitable avec la Partie 3)
t0 = time.time()
r = (sc.textFile(PATH).map(json.loads)
       .filter(lambda e: e["status"] == "CARE_STARTED")
       .map(lambda e: (e["city"], (e["wait_minutes"], 1)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
       .collect())
print("Temps RDD (lecture + parsing + agrégation) : %.2f s" % (time.time() - t0))

[Stage 17:=============================>                            (1 + 1) / 2]

Temps RDD (lecture + parsing + agrégation) : 5.98 s


----------------------------------------                                        
Exception occurred during processing of request from ('127.0.0.1', 33750)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/s